# Scraper Paralelo de Mercadona - CSV con Info Nutricional

Versión optimizada que combina:
- Paralelización del `new_notebook.ipynb`
- Funcionalidades mejoradas del `mercadona_scraper_complete.ipynb`
- Salida CSV con estructura: `id,Category,name,subtitle,price,discount_price,main_image_url,secondary_image_url,nutritional_info`

In [1]:
# IMPORTAR LIBRERÍAS
import os
import time
import json
import re
import math
import pandas as pd
import concurrent.futures
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

# Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys

# Configurar directorios
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Librerías importadas")
print(f"📁 Archivos CSV se guardarán en: {OUTPUT_DIR}")

✅ Librerías importadas
📁 Archivos CSV se guardarán en: ../data/processed


In [2]:
# FUNCIONES CORE - SELENIUM Y PARSING

def create_optimized_driver():
    """Crear driver de Chrome optimizado para scraping paralelo."""
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")
    options.add_argument("--log-level=3")
    options.add_argument("--disable-images")  # Más rápido para datos básicos
    options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36")
    
    try:
        # Usar ChromeDriver instalado localmente con Homebrew
        service = Service('/opt/homebrew/bin/chromedriver')
        driver = webdriver.Chrome(service=service, options=options)
        print("✅ Usando ChromeDriver local de Homebrew")
    except Exception as e:
        print(f"⚠️ Error con ChromeDriver local: {e}")
        try:
            # Fallback: intentar con ChromeDriverManager
            service = Service(ChromeDriverManager().install())
            driver = webdriver.Chrome(service=service, options=options)
            print("✅ Usando ChromeDriverManager como fallback")
        except Exception as e2:
            print(f"⚠️ Error con ChromeDriverManager: {e2}")
            # Último fallback: chromedriver del sistema
            try:
                driver = webdriver.Chrome(options=options)
                print("✅ Usando ChromeDriver del sistema")
            except Exception as e3:
                print(f"❌ Error creando driver: {e3}")
                raise e3
    
    driver.set_page_load_timeout(30)
    driver.implicitly_wait(5)
    return driver

def setup_mercadona_session(driver):
    """Configurar sesión de Mercadona."""
    try:
        driver.get("https://tienda.mercadona.es")
        wait = WebDriverWait(driver, 10)
        
        try:
            postal_input = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "ym-hide-content")))
            postal_input.clear()
            postal_input.send_keys("28039")
            
            submit_btn = driver.find_element(By.XPATH, "/html/body/div[1]/div[5]/div/div[2]/div/form/button")
            submit_btn.click()
            
            wait.until(EC.invisibility_of_element_located((By.CLASS_NAME, "ym-hide-content")))
        except:
            pass  # Ya configurado
        
        time.sleep(2)
        return True
    except Exception as e:
        print(f"❌ Error configurando Mercadona: {e}")
        return False

def extract_products_from_page_basic(html_content):
    """Extrae productos básicos de la vista de página CON información nutricional."""
    soup = BeautifulSoup(html_content, 'html.parser')
    products = []
    
    try:
        # Obtener categoría
        category_element = soup.select_one('h1.category-detail__title')
        category = category_element.get_text(strip=True) if category_element else "Sin categoría"
        
        # Encontrar productos
        product_links = soup.select('button.product-cell__content-link')
        
        for i, link in enumerate(product_links):
            product = {
                'product_index': i,
                'category': category,
                'name': None,
                'subtitle': None,
                'price': None,
                'discount_price': None,
                'main_image_url': None,
                'nutritional_info': None
            }
            
            try:
                # Nombre
                name_elem = link.select_one('h4[data-testid="product-cell-name"]')
                if name_elem:
                    product['name'] = name_elem.get_text(strip=True)
                
                # Subtítulo/formato
                format_elem = link.select_one('div.product-format')
                if format_elem:
                    format_spans = format_elem.find_all('span', class_='footnote1-r')
                    if format_spans:
                        product['subtitle'] = ' '.join(span.get_text(strip=True) for span in format_spans)
                
                # LÓGICA MEJORADA DE PRECIOS CON DESCUENTO
                # Buscar elementos de descuento
                discount_price_elem = link.select_one('p.product-price__unit-price--discount')
                previous_price_elem = link.select_one('p.product-price__previous-unit-price')
                regular_price_elem = link.select_one('p[data-testid="product-price"]')
                
                if discount_price_elem and previous_price_elem:
                    # Producto con descuento: precio original es el previous, precio descontado es el discount
                    product['price'] = previous_price_elem.get_text(strip=True)
                    product['discount_price'] = discount_price_elem.get_text(strip=True)
                elif regular_price_elem:
                    # Producto sin descuento: solo precio regular
                    product['price'] = regular_price_elem.get_text(strip=True)
                    product['discount_price'] = None
                
                # Imagen básica (convertir a 600x600)
                img_elem = link.select_one('img')
                if img_elem:
                    src = img_elem.get('src', '')
                    if src:
                        product['main_image_url'] = src.replace('h=300&w=300', 'h=600&w=600').replace('h=300', 'h=600').replace('w=300', 'w=600')
                
                # Información nutricional desde aria-label
                aria_label = link.get('aria-label', '')
                if aria_label:
                    product['nutritional_info'] = aria_label
                
                if product['name']:
                    products.append(product)
                    
            except Exception as e:
                print(f"⚠️ Error procesando producto {i}: {e}")
                continue
                
    except Exception as e:
        print(f"❌ Error general: {e}")
    
    return products

print("✅ Funciones core creadas")

✅ Funciones core creadas


In [3]:
# FUNCIONES DETECTOR DE NOVEDADES
from difflib import SequenceMatcher

def clean_product_name(name):
    """Limpia nombre del producto para comparación más precisa."""
    if not name:
        return ""
    
    # Convertir a minúsculas y eliminar caracteres especiales
    clean_name = re.sub(r'[^\w\s]', ' ', name.lower())
    # Eliminar espacios múltiples
    clean_name = re.sub(r'\s+', ' ', clean_name).strip()
    
    return clean_name

def calculate_similarity(name1, name2):
    """Calcula similitud entre dos nombres de productos."""
    clean1 = clean_product_name(name1)
    clean2 = clean_product_name(name2)
    
    return SequenceMatcher(None, clean1, clean2).ratio()

def extract_novelties_from_homepage(driver):
    """
    Extrae productos de la sección 'Novedades' de la página principal.
    """
    novelties = []
    
    try:
        print("🔍 Extrayendo productos de la sección Novedades...")
        
        # Ir a la página principal
        driver.get("https://tienda.mercadona.es")
        time.sleep(5)
        
        # Buscar la sección de Novedades
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'html.parser')
        
        # Buscar sección con data-testid=\"section\" que contenga \"Novedades\"
        novelties_section = None
        sections = soup.find_all('section', {'data-testid': 'section'})
        
        for section in sections:
            header = section.find('h2', class_='section__header headline1-b')
            if header and 'Novedades' in header.get_text():
                novelties_section = section
                break
        
        if not novelties_section:
            print("⚠️ No se encontró la sección de Novedades")
            return []
        
        print("✅ Sección de Novedades encontrada")
        
        # Extraer productos de la sección
        product_cells = novelties_section.find_all('div', {'data-testid': 'product-cell'})
        print(f"📦 Encontrados {len(product_cells)} productos en Novedades")
        
        for cell in product_cells:
            try:
                # Extraer nombre del producto
                name_elem = cell.find('h4', {'data-testid': 'product-cell-name'})
                if name_elem:
                    product_name = name_elem.get_text(strip=True)
                    
                    novelty_product = {
                        'name': product_name,
                        'name_clean': clean_product_name(product_name)
                    }
                    
                    novelties.append(novelty_product)
                    print(f"  📝 {product_name}")
                    
            except Exception as e:
                print(f"⚠️ Error procesando producto: {e}")
                continue
        
        print(f"✅ Extraídos {len(novelties)} productos de Novedades")
        return novelties
        
    except Exception as e:
        print(f"❌ Error extrayendo novedades: {e}")
        return []

def find_matching_products(novelties, csv_products, similarity_threshold=0.85):
    """
    Encuentra productos que coinciden entre novedades y CSV.
    """
    matches = []
    
    print(f"🔍 Buscando coincidencias (umbral similitud: {similarity_threshold})...")
    
    for novelty in novelties:
        novelty_name = novelty['name']
        best_match = None
        best_similarity = 0
        
        for idx, csv_product in csv_products.iterrows():
            csv_name = str(csv_product.get('name', ''))
            
            # Calcular similitud
            similarity = calculate_similarity(novelty_name, csv_name)
            
            if similarity > best_similarity and similarity >= similarity_threshold:
                best_similarity = similarity
                best_match = {
                    'csv_index': idx,
                    'csv_name': csv_name,
                    'novelty_name': novelty_name,
                    'similarity': similarity
                }
        
        if best_match:
            matches.append(best_match)
            print(f"  ✅ Coincidencia: '{best_match['novelty_name'][:50]}...' ↔ '{best_match['csv_name'][:50]}...' ({best_match['similarity']:.2f})")
    
    print(f"📊 Total coincidencias encontradas: {len(matches)}")
    return matches

def add_novelties_column_to_csv(csv_path, matches):
    """
    Añade columna 'novedad' al CSV y guarda el resultado.
    """
    try:
        print(f"📄 Procesando CSV para añadir novedades: {csv_path}")
        df = pd.read_csv(csv_path)
        
        # Crear backup
        backup_path = Path(csv_path).parent / f"{Path(csv_path).stem}_backup{Path(csv_path).suffix}"
        df.to_csv(backup_path, index=False)
        print(f"💾 Backup creado: {backup_path}")
        
        # Añadir columna 'novedad' (por defecto False)
        df['novedad'] = False
        
        # Marcar productos que son novedades
        for match in matches:
            df.loc[match['csv_index'], 'novedad'] = True
        
        # Actualizar el archivo original con la nueva columna
        df.to_csv(csv_path, index=False)
        
        # Estadísticas
        total_products = len(df)
        novelty_count = df['novedad'].sum()
        
        print(f"\\n📊 ESTADÍSTICAS NOVEDADES:")
        print(f"  • Total productos en CSV: {total_products}")
        print(f"  • Productos marcados como novedad: {novelty_count}")
        print(f"  • Porcentaje de novedades: {(novelty_count/total_products)*100:.1f}%")
        
        return novelty_count
        
    except Exception as e:
        print(f"❌ Error procesando CSV: {e}")
        return 0

print("✅ Funciones de detector de novedades creadas")

✅ Funciones de detector de novedades creadas


In [4]:
# FUNCIÓN PRINCIPAL - SCRAPER DE UNA PÁGINA (PARA PARALELIZACIÓN)

def scrape_single_page_csv(page_num, get_secondary_images=False):
    """Procesa una página de Mercadona y devuelve productos en formato CSV."""
    driver = create_optimized_driver()
    csv_products = []
    
    try:
        print(f"🔍 Worker procesando página {page_num}...")
        
        # Configurar Mercadona
        if not setup_mercadona_session(driver):
            return []
        
        # Navegar a la página específica
        url = f"https://tienda.mercadona.es/categories/{page_num}"
        driver.get(url)
        time.sleep(3)
        
        # Extraer productos básicos
        page_source = driver.page_source
        basic_products = extract_products_from_page_basic(page_source)
        
        if not basic_products:
            print(f"⚠️ No se encontraron productos en página {page_num}")
            return []
        
        print(f"📦 Página {page_num}: {len(basic_products)} productos encontrados")
        
        # Si no necesitamos imágenes secundarias, devolver productos básicos
        if not get_secondary_images:
            for i, product in enumerate(basic_products):
                csv_row = {
                    'id': f"{page_num}_{i+1}",  # ID único por página
                    'Category': product['category'],
                    'name': product['name'],
                    'subtitle': product['subtitle'] or '',
                    'price': product['price'] or '',
                    'discount_price': product['discount_price'] or '',
                    'main_image_url': product['main_image_url'] or '',
                    'secondary_image_url': '',  # No procesamos secundarias en modo rápido
                    'nutritional_info': product['nutritional_info'] or ''
                }
                csv_products.append(csv_row)
            
            return csv_products
        
        # MODO COMPLETO: Obtener imágenes secundarias
        for i, basic_product in enumerate(basic_products):
            try:
                # Recargar página si no es el primer producto
                if i > 0:
                    driver.get(url)
                    time.sleep(2)
                
                # Intentar obtener imagen secundaria
                secondary_url = ''
                
                try:
                    # Buscar botones de productos
                    product_buttons = driver.find_elements(By.CSS_SELECTOR, 'button[data-testid="open-product-detail"]')
                    if not product_buttons:
                        product_buttons = driver.find_elements(By.CLASS_NAME, 'product-cell__content-link')
                    
                    if i < len(product_buttons):
                        # Clic en producto
                        driver.execute_script("arguments[0].click();", product_buttons[i])
                        time.sleep(3)
                        
                        # Buscar thumbnails
                        thumbnails = driver.find_elements(By.CSS_SELECTOR, 'button.product-gallery__thumbnail')
                        if len(thumbnails) > 1:
                            # Clic en segundo thumbnail
                            driver.execute_script("arguments[0].click();", thumbnails[1])
                            time.sleep(2)
                            
                            # Obtener imagen HD
                            img_elem = driver.find_element(By.CSS_SELECTOR, 'div.image-zoomer__source img')
                            secondary_url = img_elem.get_attribute('src')
                            if secondary_url:
                                secondary_url = secondary_url.replace('h=300&w=300', 'h=600&w=600').replace('h=300', 'h=600').replace('w=300', 'w=600')
                        
                        # Cerrar modal
                        driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.ESCAPE)
                        time.sleep(1)
                        
                except Exception as e:
                    print(f"⚠️ Error obteniendo imagen secundaria para producto {i}: {e}")
                
                # Crear fila CSV
                csv_row = {
                    'id': f"{page_num}_{i+1}",
                    'Category': basic_product['category'],
                    'name': basic_product['name'],
                    'subtitle': basic_product['subtitle'] or '',
                    'price': basic_product['price'] or '',
                    'discount_price': basic_product['discount_price'] or '',
                    'main_image_url': basic_product['main_image_url'] or '',
                    'secondary_image_url': secondary_url,
                    'nutritional_info': basic_product['nutritional_info'] or ''
                }
                csv_products.append(csv_row)
                
            except Exception as e:
                print(f"❌ Error procesando producto {i} en página {page_num}: {e}")
                continue
        
        return csv_products
        
    except Exception as e:
        print(f"❌ Error en página {page_num}: {e}")
        return []
    finally:
        driver.quit()

print("✅ Función de scraping individual creada")

✅ Función de scraping individual creada


In [5]:
# SCRAPER PARALELO PRINCIPAL

def scrape_mercadona_parallel_csv(start_page=0, end_page=1000, num_workers=10, get_secondary_images=True):
    """
    Scraper paralelo que genera CSV con estructura completa.
    
    Parameters:
    -----------
    start_page : int
        Página inicial
    end_page : int
        Página final
    num_workers : int
        Número de workers paralelos
    get_secondary_images : bool
        Si obtener imágenes secundarias (más lento pero completo)
    """
    all_products = []
    
    print(f"🚀 SCRAPER PARALELO CSV INICIADO")
    print(f"📄 Estructura: id,Category,name,subtitle,price,discount_price,main_image_url,secondary_image_url,nutritional_info,novedad")
    print(f"📊 Páginas: {start_page} a {end_page} ({end_page - start_page + 1} total)")
    print(f"⚡ Workers: {num_workers}")
    print(f"📸 Imágenes secundarias: {'✅ Sí' if get_secondary_images else '❌ No (modo rápido)'}")
    print(f"🥗 Info nutricional: ✅ Incluida")
    print(f"🆕 Detección de novedades: ✅ Activada")
    print("="*60)
    
    start_time = time.time()
    
    # Lista de páginas a procesar
    pages_to_process = list(range(start_page, end_page + 1))
    
    # Usar ThreadPoolExecutor para procesamiento paralelo
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
        with tqdm(total=len(pages_to_process), desc="🔍 Scraping páginas") as pbar:
            
            # Enviar trabajos
            futures = {
                executor.submit(scrape_single_page_csv, page_num, get_secondary_images): page_num 
                for page_num in pages_to_process
            }
            
            # Procesar resultados conforme se completan
            for future in concurrent.futures.as_completed(futures):
                page_num = futures[future]
                try:
                    page_products = future.result()
                    if page_products:
                        all_products.extend(page_products)
                        print(f"✅ Página {page_num}: {len(page_products)} productos")
                    else:
                        print(f"⚠️ Página {page_num}: Sin productos")
                except Exception as e:
                    print(f"❌ Error en página {page_num}: {e}")
                
                pbar.update(1)
    
    end_time = time.time()
    
    # Estadísticas finales
    total_time = end_time - start_time
    
    print(f"\\n🎉 SCRAPING PARALELO COMPLETADO")
    print(f"⏱️ Tiempo total: {total_time:.2f} segundos ({total_time/60:.1f} minutos)")
    print(f"📦 Total productos extraídos: {len(all_products)}")
    
    if all_products:
        # Estadísticas de contenido
        products_with_images = len([p for p in all_products if p.get('main_image_url')])
        products_with_secondary = len([p for p in all_products if p.get('secondary_image_url')])
        products_with_nutrition = len([p for p in all_products if p.get('nutritional_info')])
        products_with_discount = len([p for p in all_products if p.get('discount_price')])
        
        print(f"📸 Productos con imagen principal: {products_with_images}/{len(all_products)}")
        print(f"📸 Productos con imagen secundaria: {products_with_secondary}/{len(all_products)}")
        print(f"🥗 Productos con info nutricional: {products_with_nutrition}/{len(all_products)}")
        print(f"💰 Productos con descuento: {products_with_discount}/{len(all_products)}")
        
        # Generar IDs secuenciales
        for i, product in enumerate(all_products, 1):
            product['id'] = i
        
        # Añadir columna novedad por defecto (False)
        for product in all_products:
            product['novedad'] = False
        
        # Guardar CSV temporal
        timestamp = int(time.time())
        filename = OUTPUT_DIR / f"mercadona_completo_{start_page}to{end_page}_{len(all_products)}prods_{timestamp}.csv"
        
        df = pd.DataFrame(all_products, columns=[
            'id', 'Category', 'name', 'subtitle', 'price', 'discount_price', 
            'main_image_url', 'secondary_image_url', 'nutritional_info', 'novedad'
        ])
        
        df.to_csv(filename, index=False, encoding='utf-8')
        
        print(f"\\n📄 CSV GUARDADO: {filename.name}")
        print(f"💾 Tamaño: {filename.stat().st_size / 1024 / 1024:.1f} MB")
        
        # DETECCIÓN AUTOMÁTICA DE NOVEDADES
        print(f"\\n🆕 INICIANDO DETECCIÓN AUTOMÁTICA DE NOVEDADES")
        print("=" * 60)
        
        driver = None
        try:
            # Crear driver para novedades
            print("🔧 Creando driver para detección de novedades...")
            driver = create_optimized_driver()
            
            # Configurar sesión
            if setup_mercadona_session(driver):
                # Extraer novedades
                novelties = extract_novelties_from_homepage(driver)
                
                if novelties:
                    # Encontrar coincidencias
                    matches = find_matching_products(novelties, df)
                    
                    if matches:
                        # Añadir columna de novedades al CSV
                        novelty_count = add_novelties_column_to_csv(str(filename), matches)
                        
                        print(f"\\n🎉 DETECCIÓN DE NOVEDADES COMPLETADA")
                        print(f"✅ {novelty_count} productos marcados como novedades")
                        
                        # Mostrar algunos productos marcados como novedad
                        df_updated = pd.read_csv(filename)
                        novelties_df = df_updated[df_updated['novedad'] == True]
                        
                        if len(novelties_df) > 0:
                            print(f"\\n🎯 ALGUNOS PRODUCTOS MARCADOS COMO NOVEDADES:")
                            for i, row in novelties_df.head(3).iterrows():
                                print(f"  • {row.get('name', 'Sin nombre')} ({row.get('Category', 'Sin categoría')})")
                    else:
                        print("ℹ️ No se encontraron coincidencias entre novedades y CSV")
                else:
                    print("❌ No se pudieron extraer novedades de la homepage")
            else:
                print("❌ No se pudo configurar la sesión para detección de novedades")
        
        except Exception as e:
            print(f"❌ Error en detección de novedades: {e}")
        
        finally:
            if driver:
                driver.quit()
                print("🔄 Navegador de novedades cerrado")
        
        return str(filename), all_products
    
    else:
        print("❌ No se encontraron productos")
        return None, []

print("✅ Scraper paralelo principal actualizado con detección automática de novedades")

✅ Scraper paralelo principal actualizado con detección automática de novedades


In [6]:
scrape_mercadona_parallel_csv()

🚀 SCRAPER PARALELO CSV INICIADO
📄 Estructura: id,Category,name,subtitle,price,discount_price,main_image_url,secondary_image_url,nutritional_info,novedad
📊 Páginas: 0 a 1000 (1001 total)
⚡ Workers: 10
📸 Imágenes secundarias: ✅ Sí
🥗 Info nutricional: ✅ Incluida
🆕 Detección de novedades: ✅ Activada


🔍 Scraping páginas:   0%|          | 0/1001 [00:00<?, ?it/s]

✅ Usando ChromeDriver local de Homebrew
✅ Usando ChromeDriver local de Homebrew
✅ Usando ChromeDriver local de Homebrew
🔍 Worker procesando página 4...
🔍 Worker procesando página 6...
🔍 Worker procesando página 8...
✅ Usando ChromeDriver local de Homebrew
✅ Usando ChromeDriver local de Homebrew
✅ Usando ChromeDriver local de Homebrew
🔍 Worker procesando página 1...
✅ Usando ChromeDriver local de Homebrew
✅ Usando ChromeDriver local de Homebrew
✅ Usando ChromeDriver local de Homebrew
🔍 Worker procesando página 9...
🔍 Worker procesando página 3...
🔍 Worker procesando página 5...
🔍 Worker procesando página 7...
🔍 Worker procesando página 0...
✅ Usando ChromeDriver local de Homebrew
🔍 Worker procesando página 2...
⚠️ No se encontraron productos en página 3
⚠️ No se encontraron productos en página 0
⚠️ Página 3: Sin productos
⚠️ Página 0: Sin productos
⚠️ No se encontraron productos en página 4
✅ Usando ChromeDriver local de Homebrew
✅ Usando ChromeDriver local de Homebrew
🔍 Worker procesan

('../data/processed/mercadona_completo_0to1000_4659prods_1755777584.csv',
 [{'id': 1,
   'Category': 'Vacuno',
   'name': 'Filetes de vacuno 1º B añojo para plancha o guisar',
   'subtitle': 'Bandeja 550 g aprox.',
   'price': '9,74 €',
   'discount_price': '',
   'main_image_url': 'https://prod-mercadona.imgix.net/images/49cda6c9e89270cb5db57e9af543fd60.jpg?fit=crop&h=600&w=600',
   'secondary_image_url': 'https://prod-mercadona.imgix.net/images/c597a3f06c95389e0fe2745468438af7.jpg?fit=crop&h=600&w=600',
   'nutritional_info': 'Filetes de vacuno 1º B añojo para plancha o guisar, Bandeja, 550 Gramos aprox., 9,74€ por Unidad',
   'novedad': False},
  {'id': 2,
   'Category': 'Vacuno',
   'name': 'Filetes lomo de vacuno añojo para plancha',
   'subtitle': 'Bandeja 440 g aprox.',
   'price': '9,55 €',
   'discount_price': '',
   'main_image_url': 'https://prod-mercadona.imgix.net/images/d961def65e50cdae8c037bdeab8731f9.jpg?fit=crop&h=600&w=600',
   'secondary_image_url': 'https://prod-mer